# 05 — Figures and acquisition-artifact characterization

Figures 2–6 of the article and the acquisition-artifact numbers reported in the Results.
No feature or test is recomputed: the figures read the outputs of notebooks 01–03. The raw
EDF files are read only for the signal before correction (Fig. 2b "before", Fig. 3 and the
per-record table).

| Figure | Content | Inputs |
|---|---|---|
| Fig. 2 | (a) λ₁ of all records by group and recording session, threshold and gap; (b) global theta PLV before and after correction, final sample (EC) | `theta_artifact_scan.csv`, EDF files, `features_EC.csv` |
| Fig. 3 | (a) PSD of a January 2013 control and a May–July 2013 control, with multiples of f₀; (b) phase per channel at f₀ in EDF channel order; (c) global theta PLV before correction against recording date (EC) | EDF files |
| Fig. 4 | (a) Venn diagram of the features significant in each condition; (b)–(d) Cliff's δ between pairs of conditions | `stats_<COND>.csv` |
| Fig. 5 | (a) Interaction p-values; (b) smallest attainable equivalence margin | `layer2_interaction.csv`, `layer3_margin_by_feature.csv` |
| Fig. 6 | Recording-batch effect by feature family | `batch_effect_by_feature.csv` |

**Outputs (in `FIG_DIR`):** `fig2`–`fig6` as PDF (vector) and PNG (300 dpi),
`record_table.csv` (date and theta PLV before correction of every record),
`fig6_values.csv`, `fig6_values_by_condition.csv`, `harmonic_peaks.csv`.

## Setup

In [ ]:
# Google Colab only: install dependencies and mount Google Drive.
import sys
if 'google.colab' in sys.modules:
    !pip install -q mne==1.13.2 statsmodels PyWavelets
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
import sys

REPO_DIR = '..'   # repository root (on Colab: the folder where the repository was cloned or copied)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

import json
import warnings

import matplotlib
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from matplotlib.patches import Circle
from scipy import signal, stats

from eeg_features import CHANNEL_NAMES, CONDITIONS

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')
print(f'Python {sys.version.split()[0]}')
for module in (mne, np, pd, matplotlib):
    print(f'{module.__name__:12s} {module.__version__}')

## Configuration

In [ ]:
EDF_DIR = 'path/to/original_edf_files'      # original EDF files
PREPROCESSED_DIR = 'path/to/preprocessed'   # OUTPUT_DIR of 01_preprocessing.ipynb
RESULTS_DIR = 'path/to/results'             # outputs of notebooks 02 and 03
FIG_DIR = 'path/to/figures'
os.makedirs(FIG_DIR, exist_ok=True)

EXAMPLE_AFFECTED, EXAMPLE_CLEAN = 'HC_4', 'HC_16'   # Fig. 3a-b: January 2013 and May 2013 controls
EXAMPLE_CONDITION = 'EC'
JANUARY_SESSION = ('2013-01-12', '2013-01-14')        # controls with the acquisition artifact
CONTEMPORARY_PERIOD = ('2013-04-01', '2013-07-31')    # period of the patients
LAMBDA_MIN = 0.425                                    # GED threshold, midpoint of the gap in lambda_1
ALPHA = 0.05
MARGINS = (0.20, 0.30, 0.50)                          # equivalence margins (SD), Fig. 5b
SFREQ = 256

### Style and labels

In [ ]:
# Colors follow the entity in all figures: January 2013 controls = blue, MDD = orange,
# contemporaneous controls = teal, controls from other sessions = gray.
BLUE, ORANGE, TEAL = '#2a78d6', '#eb6834', '#1baf7a'
GRAY, INK, INK2, GRID = '#a8a7a2', '#0b0b0b', '#52514e', '#e4e3df'
COLOR = {'jan': BLUE, 'contemp': TEAL, 'mdd': ORANGE, 'other': GRAY, 'nodate': '#d4d3cf'}

COL1, COL2 = 3.5, 7.16      # IEEE widths (inches): one and two columns
plt.rcParams.update({
    'font.family': 'serif', 'font.size': 8, 'axes.titlesize': 8,
    'axes.labelsize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 7, 'legend.frameon': False,
    'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'xtick.color': INK2,
    'ytick.color': INK2, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5,
    'axes.axisbelow': True, 'lines.linewidth': 1.2, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'pdf.fonttype': 42,
})

SESSION_LABEL = {'jan': 'HC Jan 2013', 'contemp': 'HC May–Jul 2013', 'other': 'HC, other sessions',
                 'nodate': 'HC (no date)', 'mdd': 'MDD'}
PATTERN_LABEL = {'all3': 'All three', 'two': 'Two', 'one': 'One only', 'none': 'None'}
FAMILY_LABEL = {'complexity_hurst': 'Hurst exponent', 'complexity_permen': 'Permutation entropy',
                'complexity_sampen': 'Sample entropy', 'complexity_higuchi_fd': 'Higuchi FD',
                'spectral_sef95': 'SEF95', 'spectral_entropy': 'Spectral entropy', 'spectral_iaf': 'IAF',
                'connectivity_plv_theta': 'Theta PLV', 'connectivity_coherence_theta': 'Theta coherence'}
BAND_LABEL = {'low_alpha': 'low alpha', 'high_alpha': 'high alpha'}


def family_label(family):
    # Display name of a family returned by eeg_features.family_of.
    if family in FAMILY_LABEL:
        return FAMILY_LABEL[family]
    p = family.split('_')
    band = lambda tokens: BAND_LABEL.get('_'.join(tokens), '_'.join(tokens))
    if p[0] == 'spectral' and p[1] in ('rel', 'abs'):
        return f"{'Relative' if p[1] == 'rel' else 'Absolute'} {band(p[2:])}"
    if p[0] == 'tf':
        return f"CWT {dict(mean='mean', std='SD', cv='CV', iqr='IQR')[p[1]]}, {band(p[2:])}"
    if p[0] == 'ratio':
        return f'{p[1].capitalize()}/{p[2]} ratio'
    if p[0] == 'connectivity':
        return f"{'PLV' if p[1] == 'plv' else 'Coherence'} {band(p[2:])}"
    if p[0] == 'asymmetry':
        return f'Asymmetry, {band(p[1:])}'
    return family


def save(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIG_DIR, f'{name}.{ext}'))
    plt.close(fig)
    print(f'saved {name}.pdf / .png')


def panel_label(ax, letter):
    ax.text(-0.14, 1.04, f'({letter})', transform=ax.transAxes, fontsize=9,
            fontweight='bold', va='bottom', ha='left', color=INK)

## Per-record table: recording date and theta PLV before correction

For every record of `theta_artifact_scan.csv`, the raw EDF file is read with the 19 channels,
average reference and a 1–45 Hz filter (the signal before the GED step), and the global theta
PLV is computed as in the features: mean over the 171 channel pairs, linear-phase FIR 4–8 Hz
and Hilbert phase, in 10-s blocks averaged across blocks (PLV is biased upward in short
segments, so a whole-record PLV would not be comparable with the values after correction).
The table is cached in `FIG_DIR/record_table.csv`.

In [ ]:
def load_edf(path, preprocess=True):
    # 19 channels; returns the raw object and the channel order in the EDF file.
    raw = mne.io.read_raw_edf(path, preload=True, verbose='ERROR')
    mapping = {}
    for ch in raw.ch_names:
        name = ch.replace('EEG', '').replace('-LE', '').replace('-REF', '').strip()
        for c in CHANNEL_NAMES:
            if name.lower() == c.lower():
                mapping[ch] = c
    edf_order = [mapping[ch] for ch in raw.ch_names if ch in mapping]
    raw.pick(list(mapping)).rename_channels(mapping)
    if preprocess:
        raw.set_eeg_reference('average', verbose='ERROR')
        raw.filter(1, 45, verbose='ERROR')
    return raw, edf_order


def global_theta_plv(x, fs, block_s=10):
    b = signal.firwin(int(3 * fs / 4) | 1, [4, 8], pass_zero=False, fs=fs)
    phase = np.angle(signal.hilbert(signal.filtfilt(b, 1, x, axis=-1), axis=-1))
    n, trim = int(block_s * fs), int(fs / 4)
    iu = np.triu_indices(len(x), 1)
    vals = []
    for i0 in range(0, phase.shape[1] - n + 1, n):
        z = np.exp(1j * phase[:, i0 + trim:i0 + n - trim])
        vals.append((np.abs(z @ z.conj().T) / z.shape[1])[iu].mean())
    return float(np.mean(vals))


def session(subject, date):
    # jan = January 2013 session; contemp = controls of the patients' period; other = other sessions.
    if not subject.startswith('HC'):
        return 'mdd'
    if date is None or pd.isna(date):
        return 'nodate'
    if pd.Timestamp(JANUARY_SESSION[0]) <= date <= pd.Timestamp(JANUARY_SESSION[1]) + pd.Timedelta(days=1):
        return 'jan'
    if pd.Timestamp(CONTEMPORARY_PERIOD[0]) <= date <= pd.Timestamp(CONTEMPORARY_PERIOD[1]) + pd.Timedelta(days=1):
        return 'contemp'
    return 'other'


scan = pd.read_csv(os.path.join(PREPROCESSED_DIR, 'theta_artifact_scan.csv'))
cache = os.path.join(FIG_DIR, 'record_table.csv')
if os.path.exists(cache):
    records = pd.read_csv(cache, parse_dates=['date'])
else:
    rows = []
    for r in scan.itertuples():
        raw, _ = load_edf(os.path.join(EDF_DIR, r.file))
        rows.append({'subject_id': r.subject_id, 'condition': r.condition,
                     'date': pd.Timestamp(raw.info['meas_date']).tz_localize(None),
                     'plv_theta_before': global_theta_plv(raw.get_data() * 1e6, raw.info['sfreq'])})
    records = pd.DataFrame(rows)
    records.to_csv(cache, index=False)

scan = scan.merge(records, on=['subject_id', 'condition'], how='left')
scan['session'] = [session(s, d) for s, d in zip(scan.subject_id, scan.date)]
scan['corrected'] = scan.lambda1 >= LAMBDA_MIN
sample = set(pd.read_csv(os.path.join(RESULTS_DIR, 'features_EC.csv'), usecols=['subject']).subject)
scan['final_sample'] = scan.subject_id.isin(sample)

# Gap in the lambda_1 distribution (as in Step 0 of 01_preprocessing.ipynb)
lam1 = np.sort(scan.lambda1.values)
gaps = np.diff(lam1)
gaps[:2] = gaps[-2:] = 0
i = int(np.argmax(gaps))
GAP = (lam1[i], lam1[i + 1])
assert GAP[0] < LAMBDA_MIN < GAP[1], 'LAMBDA_MIN is outside the gap'
print(f'{len(scan)} records | gap in lambda_1: {GAP[0]:.3f} to {GAP[1]:.3f} | corrected: {int(scan.corrected.sum())} '
      f'| final sample: {len(sample)} participants')
print(scan.groupby(['session', 'corrected']).size().unstack(fill_value=0).to_string())

## Fig. 2 — Artifact detection and correction

In [ ]:
def fig2():
    fig, axs = plt.subplots(1, 2, figsize=(COL2, 2.4))

    # (a) lambda_1, stacked by group and session
    ax = axs[0]
    bins = np.arange(0, 1.0001, 0.025)
    order = [s for s in ['jan', 'contemp', 'other', 'nodate', 'mdd'] if s in set(scan.session)]
    ax.hist([scan.loc[scan.session == s, 'lambda1'] for s in order], bins=bins, stacked=True,
            color=[COLOR[s] for s in order],
            label=[f'{SESSION_LABEL[s]} (n = {(scan.session == s).sum()})' for s in order],
            edgecolor='white', linewidth=0.6)
    ax.axvspan(*GAP, color=GRID, alpha=0.5, lw=0, zorder=0)
    ax.axvline(LAMBDA_MIN, color=INK, lw=0.9, ls='--')
    ax.text(LAMBDA_MIN + 0.01, ax.get_ylim()[1] * 0.55, f'λ = {LAMBDA_MIN:.3g}', fontsize=7,
            va='center', ha='left', color=INK)
    ax.set_xlabel('λ₁ (power fraction in f₀ ± 0.5 Hz)'); ax.set_ylabel('Recordings')
    ax.legend(loc='upper right')
    panel_label(ax, 'a')

    # (b) global theta PLV before (EDF) and after correction (features), final sample
    ax = axs[1]
    after = pd.read_csv(os.path.join(RESULTS_DIR, f'features_{EXAMPLE_CONDITION}.csv'),
                        usecols=['subject', 'plv_global_theta'])
    d = scan[scan.condition == EXAMPLE_CONDITION].merge(
        after.rename(columns={'subject': 'subject_id', 'plv_global_theta': 'plv_theta_after'}), on='subject_id')
    for ses, x0 in [('jan', 0), ('contemp', 1), ('mdd', 2)]:
        s = d[d.session == ses]
        for _, r in s.iterrows():
            ax.plot([x0 - 0.15, x0 + 0.15], [r.plv_theta_before, r.plv_theta_after],
                    color=COLOR[ses], lw=0.6, alpha=0.6)
        ax.scatter(np.full(len(s), x0 - 0.15), s.plv_theta_before, s=10, color=COLOR[ses], zorder=3)
        ax.scatter(np.full(len(s), x0 + 0.15), s.plv_theta_after, s=10, color=COLOR[ses], marker='D', zorder=3)
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels([SESSION_LABEL[s] for s in ('jan', 'contemp', 'mdd')])
    ax.set_ylabel(f'Global theta PLV ({EXAMPLE_CONDITION})')
    ax.text(0.98, 0.98, '● before   ◆ after', transform=ax.transAxes, ha='right', va='top', fontsize=7, color=INK2)
    panel_label(ax, 'b')
    fig.tight_layout(w_pad=1.5)
    save(fig, 'fig2')


fig2()

## Fig. 3 — Acquisition artifact

In [ ]:
def edf_path(subject, condition):
    return os.path.join(EDF_DIR, scan.set_index(['subject_id', 'condition']).loc[(subject, condition), 'file'])


def fig3():
    fig, axs = plt.subplots(1, 3, figsize=(COL2, 2.4), gridspec_kw={'width_ratios': [1, 1.15, 1.25]})
    path_a, path_c = edf_path(EXAMPLE_AFFECTED, EXAMPLE_CONDITION), edf_path(EXAMPLE_CLEAN, EXAMPLE_CONDITION)

    # (a) PSD averaged over channels, with tick marks at multiples of f0
    ax = axs[0]
    for path, subject, key in [(path_a, EXAMPLE_AFFECTED, 'jan'), (path_c, EXAMPLE_CLEAN, 'contemp')]:
        raw, _ = load_edf(path)
        fs = raw.info['sfreq']
        f, P = signal.welch(raw.get_data() * 1e6, fs=fs, nperseg=int(4 * fs), axis=-1)
        Pm = P.mean(0)
        sel = (f >= 1) & (f <= 30)
        ax.semilogy(f[sel], Pm[sel], color=COLOR[key], label=f'{subject} ({SESSION_LABEL[key]})')
        if path == path_a:
            search = (f >= 4.5) & (f <= 6.5)
            f0 = f[search][np.argmax(Pm[search])]
    ax.axvspan(4.5, 6.5, color=GRID, alpha=0.6, lw=0)
    for k in range(2, int(30 // f0) + 1):     # narrow peaks not reached by the removal at f0 ± 0.5 Hz
        ax.axvline(k * f0, ymin=0.92, ymax=1.0, color=COLOR['jan'], lw=0.8)
    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('PSD (µV²/Hz)')
    ax.legend(loc='lower left', handlelength=1.2, bbox_to_anchor=(0, 1.0), borderaxespad=0.1)
    panel_label(ax, 'a')

    # (b) phase relative to the first channel at f0, in EDF channel order (raw signal, no re-referencing).
    # Wrapped to (-180, 180]; unwrapping would turn an alternation between a few values into a ramp.
    ax = axs[1]
    raw_b, edf_order = load_edf(path_a, preprocess=False)
    raw_b.filter(1, 45, verbose='ERROR')
    x = raw_b.get_data(picks=edf_order) * 1e6
    fs = raw_b.info['sfreq']
    f, Pxy = signal.csd(x, x[0][None, :], fs=fs, nperseg=int(4 * fs), axis=-1)
    k = np.argmin(np.abs(f - f0))
    phase = np.degrees(np.angle(Pxy[:, k] * np.conj(Pxy[0, k])))
    ax.plot(range(len(edf_order)), phase, color=BLUE, marker='o', ms=3.5, lw=0.8)
    ax.set_ylim(-195, 195); ax.set_yticks([-180, -90, 0, 90, 180])
    ax.set_xticks(range(len(edf_order)))
    ax.set_xticklabels(edf_order, rotation=90, fontsize=6)
    ax.set_xlabel(f'Channel (EDF order), {EXAMPLE_AFFECTED}')
    ax.set_ylabel(f'Phase at {f0:.2f} Hz (deg)')
    panel_label(ax, 'b')

    # (c) global theta PLV before correction against recording date, all records
    ax = axs[2]
    d = scan[scan.condition == EXAMPLE_CONDITION]
    for ses in [s for s in ['jan', 'contemp', 'other', 'nodate', 'mdd'] if s in set(d.session)]:
        s = d[d.session == ses]
        ax.scatter(s.date, s.plv_theta_before, s=14, color=COLOR[ses], label=SESSION_LABEL[ses],
                   edgecolor='white', linewidth=0.6, zorder=3)
    ax.set_ylabel(f'Global theta PLV ({EXAMPLE_CONDITION}, before)')
    ax.set_xlabel('Recording date')
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%m/%Y'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    ax.legend(loc='center right', handletextpad=0.2)
    panel_label(ax, 'c')
    fig.tight_layout(w_pad=1.2)
    save(fig, 'fig3')
    print(f'f0 of {EXAMPLE_AFFECTED}: {f0:.2f} Hz; multiples marked: '
          + ', '.join(f'{k * f0:.1f}' for k in range(2, int(30 // f0) + 1)))


fig3()

## Fig. 4 — Descriptive detection pattern

In [ ]:
def fig4():
    st = {c: pd.read_csv(os.path.join(RESULTS_DIR, f'stats_{c}.csv')).set_index('feature').sort_index()
          for c in CONDITIONS}
    S = pd.DataFrame({c: st[c].significant.astype(bool) for c in st})
    D = pd.DataFrame({c: st[c].cliffs_delta for c in st})
    n = S.sum(axis=1)
    pattern = np.select([n == 3, n == 2, n == 1], ['all3', 'two', 'one'], 'none')
    style = {'none': (GRAY, 4, 0.35), 'all3': (BLUE, 7, 0.85), 'two': (ORANGE, 7, 0.85), 'one': (TEAL, 9, 0.95)}
    threshold = {c: D.loc[S[c], c].abs().min() for c in D}       # smallest significant |delta|

    fig, grid = plt.subplots(2, 2, figsize=(COL2, 6.0))
    ax_v, axs = grid[0, 0], [grid[0, 1], grid[1, 0], grid[1, 1]]

    # (a) Venn diagram: neutral circles; each count has the color of its detection pattern
    key = S.apply(lambda r: '+'.join(c for c in S if r[c]) or 'none', axis=1).value_counts()
    k = {c: int(key.get(c, 0)) for c in ('EC', 'EO', 'TASK', 'EC+EO', 'EC+TASK', 'EO+TASK', 'EC+EO+TASK')}
    total = {c: int(S[c].sum()) for c in S}
    ax_v.set_xlim(-2.25, 2.25); ax_v.set_ylim(-2.25, 2.1); ax_v.set_aspect('equal'); ax_v.axis('off')
    r, d = 1.15, 0.72
    for x, y in [(-d, d * 0.58), (d, d * 0.58), (0, -d * 0.67)]:
        ax_v.add_patch(Circle((x, y), r, fc=GRID, alpha=0.35, ec=INK2, lw=0.9))
    for c, (x, y, ha) in {'EC': (-1.5, 1.8, 'right'), 'EO': (1.5, 1.8, 'left'), 'TASK': (0, -2.08, 'center')}.items():
        ax_v.text(x, y, f'{c} (n = {total[c]})', ha=ha, va='center', fontsize=7.5, fontweight='bold', color=INK)
    pos = {'EC': (-1.25, 0.85), 'EO': (1.25, 0.85), 'TASK': (0, -1.4), 'EC+EO': (0, 0.98),
           'EC+TASK': (-0.78, -0.42), 'EO+TASK': (0.78, -0.42), 'EC+EO+TASK': (0, 0.08)}
    for region, (x, y) in pos.items():
        color = style['all3' if region.count('+') == 2 else 'two' if '+' in region else 'one'][0]
        ax_v.text(x, y, f'{k[region]}', ha='center', va='center',
                  fontsize=10 if region.count('+') == 2 else 9, fontweight='bold', color=color)

    # (b)-(d) Cliff's delta in one condition against another
    for ax, (a, b) in zip(axs, [('EC', 'EO'), ('EC', 'TASK'), ('EO', 'TASK')]):
        for p in ['none', 'all3', 'two', 'one']:
            m = pattern == p
            color, size, alpha = style[p]
            ax.scatter(D.loc[m, a], D.loc[m, b], s=size, color=color, alpha=alpha, lw=0,
                       label=f'{PATTERN_LABEL[p]} (n = {m.sum()})', zorder=2 if p == 'none' else 3)
        for v in (-1, 1):
            ax.axvline(v * threshold[a], color=INK2, lw=0.6, ls=':')
            ax.axhline(v * threshold[b], color=INK2, lw=0.6, ls=':')
        ax.plot([-1, 1], [-1, 1], color=INK2, lw=0.6)
        ax.text(0.04, 0.96, f'r = {np.corrcoef(D[a], D[b])[0, 1]:.2f}', transform=ax.transAxes,
                va='top', fontsize=7, color=INK)
        ax.set_xlabel(f"Cliff's δ in {a}"); ax.set_ylabel(f"Cliff's δ in {b}")
        ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_aspect('equal')
        ax.set_xticks([-1, -0.5, 0, 0.5, 1]); ax.set_yticks([-1, -0.5, 0, 0.5, 1])
    h, l = axs[0].get_legend_handles_labels()
    order = [1, 2, 3, 0]
    fig.legend([h[i] for i in order], [l[i] for i in order], loc='lower center', ncol=4,
               title='Significant (FDR) in', title_fontsize=7, markerscale=1.6, handletextpad=0.1,
               bbox_to_anchor=(0.5, 0.0))
    for ax, letter in zip([ax_v] + axs, 'abcd'):
        panel_label(ax, letter)
    fig.tight_layout(w_pad=1.5, h_pad=1.5, rect=(0, 0.07, 1, 1))
    save(fig, 'fig4')

    single = S.index[n == 1]
    others = np.concatenate([np.abs(D.loc[f, [c for c in D if not S.loc[f, c]]].values) for f in single])
    # (feature, other condition) pairs with the sign of the condition where the feature is significant
    same_sign = np.mean(np.concatenate([np.sign(D.loc[f, [c for c in D if not S.loc[f, c]]].values)
                                        == np.sign(D.loc[f, S.loc[f]].iloc[0]) for f in single]))
    print('Smallest significant |δ|: ' + ', '.join(f'{c} {v:.2f}' for c, v in threshold.items()))
    print('Pearson r of δ between conditions: ' + ', '.join(
        f'{a}-{b} {np.corrcoef(D[a], D[b])[0, 1]:.2f}' for a, b in [('EC', 'EO'), ('EC', 'TASK'), ('EO', 'TASK')]))
    print(f'Features significant in one condition only ({len(single)}): median |δ| in that condition '
          f'{np.median([abs(D.loc[f, S.loc[f]].iloc[0]) for f in single]):.2f}, in the other two '
          f'{np.median(others):.2f}; same sign in the other conditions: {same_sign:.0%}')


fig4()

## Fig. 5 — Precision of the formal tests

(b) uses the smallest attainable margin without correction (`delta_min_raw`, the largest of the
three contrasts) computed in `03_condition_dependence.ipynb`. The minimum detectable effect is
the median, over features and contrasts, of $(t_{0.975;\nu} + t_{0.80;\nu})\,\mathrm{SE}$.

In [ ]:
def fig5():
    m = pd.read_csv(os.path.join(RESULTS_DIR, 'layer2_interaction.csv'))
    dmin = pd.read_csv(os.path.join(RESULTS_DIR, 'layer3_margin_by_feature.csv')).delta_min_raw.dropna().values
    p = m.p_interaction_perm.dropna().values
    SE = m[[f'se_{c}_z' for c in ('EO_vs_EC', 'TASK_vs_EC', 'TASK_vs_EO')]].to_numpy(float)
    nu = m.df_resid.to_numpy(float)
    mde = float(np.median((stats.t.ppf(0.975, nu) + stats.t.ppf(0.80, nu))[:, None] * SE))
    half_width = float(np.median(stats.t.ppf(1 - ALPHA, nu)[:, None] * SE))

    fig, axs = plt.subplots(1, 2, figsize=(COL2, 2.4))
    ax = axs[0]
    ax.hist(p, bins=np.linspace(0, 1, 21), color=BLUE, edgecolor='white', linewidth=0.6)
    ax.axhline(len(p) / 20, color=INK, lw=0.9, ls='--')
    ax.text(0.99, len(p) / 20 + 1, 'expected under H₀', ha='right', va='bottom', fontsize=7, color=INK)
    ax.set_xlabel('Group × Condition interaction p (permutation)'); ax.set_ylabel('Features')
    ax.text(0.98, 0.98, f'p < 0.05: {(p < 0.05).sum()} (expected {0.05 * len(p):.0f})\n'
            f'smallest q (FDR) = {m.p_interaction_fdr.min():.2f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=7, color=INK)
    panel_label(ax, 'a')

    ax = axs[1]
    ax.hist(dmin, bins=np.arange(0, np.ceil(dmin.max() * 10) / 10 + 0.05, 0.05),
            color=BLUE, edgecolor='white', linewidth=0.6)
    for d in MARGINS:
        ax.axvline(d, color=INK2, lw=0.8, ls=':')
    top = ax.get_ylim()[1]
    ax.text(np.mean(MARGINS[:2]), top * 1.01, ' / '.join(f'{d:.2f}' for d in MARGINS[:2]),
            ha='center', va='bottom', fontsize=6.5, color=INK2)
    ax.text(MARGINS[2], top * 1.01, f'{MARGINS[2]:.2f}', ha='center', va='bottom', fontsize=6.5, color=INK2)
    ax.axvline(mde, color=ORANGE, lw=1.1)
    ax.text(mde + 0.03, top * 0.92, f'MDE {mde:.2f}', ha='left', va='top', fontsize=7, color=INK)
    ax.set_xlabel('Smallest margin Δ (SD)'); ax.set_ylabel('Features')
    ax.text(0.98, 0.8, f'median 90% CI\nhalf-width = {half_width:.2f} SD', transform=ax.transAxes,
            ha='right', va='top', fontsize=7, color=INK)
    panel_label(ax, 'b')
    fig.tight_layout(w_pad=1.5)
    save(fig, 'fig5')


fig5()

## Fig. 6 — Recording-batch effect by feature family

Cliff's δ of `03_condition_dependence.ipynb` (`batch_effect_by_feature.csv`): in each
condition, the features significant in that condition; the (condition, feature) pairs of the
three conditions are pooled and summarized by the median per family. Families with at least
`MIN_PAIRS` pairs are shown, ordered by |δ| of MDD versus all controls (at most `MAX_FAMILIES`).

In [ ]:
MIN_PAIRS, MAX_FAMILIES = 4, 16


def fig6():
    t = pd.read_csv(os.path.join(RESULTS_DIR, 'batch_effect_by_feature.csv'))
    cols = ['delta_group', 'delta_mdd_vs_hc_other', 'delta_batch']
    by_cond = t.groupby(['family', 'condition'])[cols].median().round(2)
    by_cond['n'] = t.groupby(['family', 'condition']).size()
    by_cond.to_csv(os.path.join(FIG_DIR, 'fig6_values_by_condition.csv'))

    fam = t.groupby('family')[cols].median().assign(n=t.groupby('family').size())
    fam = fam[fam.n >= MIN_PAIRS]
    fam = fam.reindex(fam.delta_group.abs().sort_values(ascending=False, kind='stable').index).head(MAX_FAMILIES).iloc[::-1]
    fam.to_csv(os.path.join(FIG_DIR, 'fig6_values.csv'))

    fig, ax = plt.subplots(figsize=(COL1, 0.2 * len(fam) + 1.3))
    y = np.arange(len(fam))
    for i, (_, r) in enumerate(fam.iterrows()):
        ax.plot([r.delta_group, r.delta_mdd_vs_hc_other], [i, i], color=GRID, lw=2.2, zorder=1, solid_capstyle='round')
    ax.scatter(fam.delta_group, y, s=24, color=INK2, label='MDD vs. all HC', zorder=3)
    ax.scatter(fam.delta_mdd_vs_hc_other, y, s=24, color=TEAL, marker='D', label='MDD vs. HC May–Jul 2013',
               zorder=3, edgecolor='white', linewidth=0.6)
    ax.scatter(fam.delta_batch, y, s=28, color=BLUE, marker='x', label='HC Jan vs. HC May–Jul (batch)', zorder=3)
    ax.axvline(0, color=INK2, lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels([family_label(f) for f in fam.index])
    ax.set_xlim(-1.05, 1.05); ax.set_xlabel("Cliff's δ (median over significant features)")
    ax.grid(axis='y', visible=False)
    ax.legend(loc='lower center', bbox_to_anchor=(0.4, 1.0), ncol=1, handletextpad=0.2, borderaxespad=0.2)
    fig.tight_layout()
    save(fig, 'fig6')
    print('Families shown (bottom to top):', ', '.join(fam.index))


fig6()

## Acquisition-artifact numbers reported in the Results

### Corrected records by session and theta PLV before correction

In [ ]:
hc = scan[scan.subject_id.str.startswith('HC')]
print(f'Corrected records: {int(scan.corrected.sum())} of {len(scan)} '
      f'({int(hc.corrected.sum())} of {len(hc)} control records, '
      f'{int(scan.corrected.sum() - hc.corrected.sum())} of {len(scan) - len(hc)} patient records)')
print(f'Control records of the January session: {int((hc.session == "jan").sum())}, '
      f'corrected: {int((hc.corrected & (hc.session == "jan")).sum())}')
cols = ['subject_id', 'condition', 'date', 'lambda1', 'lambda2', 'f0', 'plv_theta_before', 'final_sample']
outside = hc[hc.corrected & (hc.session != 'jan')]
print('Corrected control records outside the January session:')
print(outside[cols].to_string(index=False) if len(outside) else '  none')
print('\nFirst recording date of the controls in the final sample:')
print(hc[hc.final_sample].groupby('subject_id').date.min().sort_values().dt.date.to_string())

c = scan[scan.corrected]
print(f'\nCorrected records: f0 {c.f0.min():.1f}-{c.f0.max():.1f} Hz (median {c.f0.median():.1f}) | '
      f'lambda_2 {c.lambda2.min():.2f}-{c.lambda2.max():.2f} | records with lambda_2 >= LAMBDA_MIN: '
      f'{int((c.lambda2 >= LAMBDA_MIN).sum())}')
print('\nGlobal theta PLV before correction (median, IQR, range):')
for name, g in (('corrected', scan[scan.corrected]), ('not corrected', scan[~scan.corrected])):
    q = g.plv_theta_before.quantile([0.25, 0.5, 0.75])
    print(f'  {name:14s} n = {len(g):3d}: {q[0.5]:.2f} ({q[0.25]:.2f}-{q[0.75]:.2f}), '
          f'min {g.plv_theta_before.min():.2f}, max {g.plv_theta_before.max():.2f}')
cols = ['subject_id', 'condition', 'lambda1', 'plv_theta_before', 'final_sample']
print('Records without correction with the highest PLV:')
print(scan[~scan.corrected].nlargest(5, 'plv_theta_before')[cols].to_string(index=False))
print('Corrected records with the lowest PLV:')
print(scan[scan.corrected].nsmallest(5, 'plv_theta_before')[cols].to_string(index=False))

### Power after correction

Ratio after/before of the theta (4–8 Hz) and alpha (8–13 Hz) power, from
`processing_log.json` of `01_preprocessing.ipynb`, against λ₁.

In [ ]:
with open(os.path.join(PREPROCESSED_DIR, 'processing_log.json'), encoding='utf-8') as fh:
    log = json.load(fh)
ratios = pd.DataFrame([{'subject_id': lg['subject_id'], 'condition': lg['condition'], **lg['theta_artifact']}
                       for lg in log if lg.get('theta_artifact')])
ratios = ratios[ratios.k > 0].merge(scan[['subject_id', 'condition', 'lambda1']], on=['subject_id', 'condition'])
print(f'Corrected records in the log: {len(ratios)}')
print(f'Theta power after/before: median {ratios.theta_ratio.median():.2f} '
      f'({ratios.theta_ratio.min():.2f}-{ratios.theta_ratio.max():.2f}) | '
      f'alpha power after/before: {ratios.alpha_ratio.min():.3f}-{ratios.alpha_ratio.max():.3f}')
print(f'Spearman rho (lambda_1, theta ratio) = {stats.spearmanr(ratios.lambda1, ratios.theta_ratio)[0]:.2f}')
ratios['lambda1_range'] = pd.cut(ratios.lambda1, [LAMBDA_MIN, 0.6, 0.7, 0.8, 0.9, 1.0])
print(ratios.groupby('lambda1_range', observed=True).theta_ratio.describe()[['count', 'min', '50%', 'max']].round(2).to_string())

### Peaks at multiples of f₀ in the corrected data

PSD of the preprocessed windows (after artifact removal and ICA), averaged over windows and
channels (0.5-Hz resolution), for every record of the final sample. For each multiple
$k f_0$ ($k = 1$–6, below 40 Hz), the peak-to-flank ratio is the maximum PSD within ±0.5 Hz
divided by the median PSD 1.5–3 Hz away. The threshold is the 95th percentile of the ratio in
the records without correction. The fraction of a band attributable to the peaks is the PSD
above the flank median within ±0.5 Hz of each $k f_0$ ($k = 2$–6) in the band, divided by the
band power.

In [ ]:
def record_psd(condition, subject):
    w = np.load(os.path.join(PREPROCESSED_DIR, f'classical_{condition}', f'{subject}_windows.npy')) * 1e6
    f, P = signal.welch(w, fs=SFREQ, nperseg=w.shape[1], axis=1)
    return f, P.mean(axis=(0, 2))


def peak_ratio(f, P, fc):
    flank = (np.abs(f - fc) >= 1.5) & (np.abs(f - fc) <= 3.0)
    return P[np.abs(f - fc) <= 0.5].max() / np.median(P[flank])


def peak_fraction(f, P, f0, band, ks):
    sel = (f >= band[0]) & (f < band[1])
    excess = 0.0
    for k in ks:
        fc = k * f0
        if band[0] <= fc < band[1]:
            flank = (np.abs(f - fc) >= 1.5) & (np.abs(f - fc) <= 3.0)
            excess += np.clip(P[np.abs(f - fc) <= 0.5] - np.median(P[flank]), 0, None).sum()
    return excess / P[sel].sum()


KS = range(2, 7)
BANDS_FRACTION = {'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 45)}
rows = []
for r in scan[scan.final_sample].itertuples():
    f, P = record_psd(r.condition, r.subject_id)
    row = {'subject_id': r.subject_id, 'condition': r.condition, 'corrected': r.corrected, 'f0': r.f0,
           'peak_f0': peak_ratio(f, P, r.f0)}
    row.update({f'peak_{k}f0': peak_ratio(f, P, k * r.f0) for k in KS if k * r.f0 < 40})
    row.update({f'fraction_{b}': peak_fraction(f, P, r.f0, lim, KS) for b, lim in BANDS_FRACTION.items()})
    rows.append(row)
harm = pd.DataFrame(rows)
harm.to_csv(os.path.join(FIG_DIR, 'harmonic_peaks.csv'), index=False)

reference = harm[~harm.corrected]
jan = harm[harm.subject_id.str.startswith('HC') & harm.corrected]
cols = [c for c in harm.columns if c.startswith('peak_')]
limit = reference[cols].quantile(0.95)
above = (jan[cols] > limit).groupby(jan.subject_id).any()
print(f'Reference: {len(reference)} records without correction | corrected control records: '
      f'{len(jan)} ({jan.subject_id.nunique()} participants)')
print(pd.DataFrame({'threshold_p95': limit.round(2),
                    'participants_above_in_any_condition': above.sum()}).to_string())
print('\nFraction of the band power attributable to the peaks at k·f0 (median; maximum):')
for b in BANDS_FRACTION:
    c = f'fraction_{b}'
    print(f'  {b:6s} corrected controls {jan[c].median():.3f} ({jan[c].max():.3f}) | '
          f'records without correction {reference[c].median():.3f} ({reference[c].max():.3f})')